# Benford's Law Digital Audit for SEC Filings

Welcome to the **Benford's Law Digital Audit** project exploration notebook. In this walkthrough, we will demonstrate how to apply forensic accounting techniques to unstructured financial data at scale.

## Section 1: What is Benford's Law?

Benford's Law, also known as the Law of First Digits, is an observation that in many naturally occurring collections of numbers, the leading digit is likely to be small. For example, the number 1 appears as the leading digit about 30% of the time, while 9 appears as the leading digit less than 5% of the time.

This counterintuitive phenomenon is widely used in forensic accounting to detect potential anomalies, data manipulation, or outright fraud in financial statements. If a company's financial figures deviate significantly from Benford's expected distribution, it warrants closer inspection.

In [ ]:
from IPython.display import Image
# Display the theoretical Benford curve generated by our visualizer
Image(filename='../outputs/charts/benford_curve.png')

## Section 2: Data Collection Methodology

To analyze a company's financial data, we extract numeric values from their SEC 10-K filings. The `sec-edgar-downloader` rapidly fetches the raw HTML filings. We then use `BeautifulSoup` and regex to strip HTML tags and extract valid, non-trivial numbers (excluding years, page numbers, and percentages). 

Once numbers are extracted, we take their first significant digit and aggregate these counts across a 10-year span to build a robust sample size.

In [ ]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.downloader import ALL_TICKERS

print(f"Target Companies: {len(ALL_TICKERS)}\n")
print("Sample of tickers analyzed:", ALL_TICKERS[:10])

## Section 3: The Suspicion Leaderboard & Heatmap

We calculate conformity using standard forensic metrics:
1. **Chi-Square Statistic** (goodness-of-fit test)
2. **Mean Absolute Deviation (MAD)** (absolute magnitude of difference)
3. **Z-Scores** (identifying statistically significant deviations in individual digits)

The companies are ranked using a composite **Suspicion Score** (40% Chi-Square, 40% MAD, 20% Flagged Digits). Below, we load the resulting leaderboard.

In [ ]:
ldb_path = Path("../outputs/leaderboard.csv")
if ldb_path.exists():
    df = pd.read_csv(ldb_path)
    display(df[['rank', 'ticker', 'sector', 'suspicion_score', 'mad', 'conformity_label']].head(10))
else:
    print("Leaderboard data not generated yet. Run `python main.py --all` to populate.")

A heatmap visually exposes the specific digits that are driving a company's anomaly score. The deeper the red, the higher the Z-score for that digit.

In [ ]:
Image(filename='../outputs/charts/leaderboard_heatmap.png')

## Section 4: Deep Dive on the Most Suspicious Company

Let's look closely at the company that ranked #1 on our Suspicion Leaderboard. The bar chart compares their observed first digits against the expected Benford percentages. Notice how some bars vastly exceed or fall short of the theoretical curve.

In [ ]:
if ldb_path.exists() and not df.empty:
    top_ticker = df.iloc[0]['ticker']
    display(Image(filename=f"../outputs/charts/company_overlay_{top_ticker}.png"))
else:
    print("Please run the pipeline to view the overlay chart.")

## Section 5: Honest Limitations & Next Steps

### What Deviation Means (and What it Doesn't)
It is crucial to understand that nonconformity with Benford's Law **does not constitute proof of fraud**. It acts purely as a smoke alarm. Deviations can occur due to perfectly legitimate reasons, such as structural changes in the business, rounding rules, or the inherent constraints of financial reporting (e.g., maximum limits, price points). 

### Sample Size Caveats
Benford's Law works best on datasets spanning multiple orders of magnitude. While 10 years of 10-K filings provide ample numbers, individual anomalies can still surface randomly. The Z-score test flags values at a 95% confidence internal, meaning 5% of flagged digits might just be false positives due to natural variance.

### Next Steps for Research
1. **Time-Series Analysis**: Tracking how a company's MAD score changes year over year could provide early warning signals before a crisis.
2. **Second-Digit Analysis**: Extending the tool to examine the second or first-two digits provides a much more robust, granular forensic test.
3. **International Filings**: Expanding the downloader to fetch data from non-US indexes.